In [89]:
from dotenv import load_dotenv

load_dotenv()
print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [90]:
from langchain_ollama import ChatOllama, OllamaLLM
from langchain.agents import create_agent
from langchain_core.tools import tool

In [91]:
model = ChatOllama(model="llama3.2", temperature=0.5, verbose=True)

@tool("calculator", return_direct=True)
def calculator(query) -> str:
    """A simple calculator tool that can evaluate basic math expressions."""
    try:
        print(f"Invoking calculator with query: {query}")
        if isinstance(query, dict):
            expression = query.get('expression', '0')
        else:
            expression = str(query)
        print(f"Evaluating expression: {expression}")
        # WARNING: Using eval can be dangerous in production. This is just for demonstration purposes.
        result = eval(expression)
        return f"The result of {expression} is: {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"  
    

In [92]:
@tool("voter_helper", return_direct=True)
def voter_helper(age: int) -> str:
    """A simple tool that provides information if person with certain age is eligible for voting or not."""
    print(f"Invoking voter_helper with query: {age}")
    if age >= 18:
        return(f"Age {age} is eligible for voting.")
    else:
        return(f"Age {age} is not eligible for voting.")


In [93]:
agent = create_agent(model=model, tools=[calculator, voter_helper])

def ask_agent(query: str) -> str:
    print("\n*************************************\n")
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

In [94]:
print(ask_agent("What is 2 + 2?"))
print(ask_agent("What is 10 * 5?"))
print(ask_agent("(3 + 4) * 2?"))
print(ask_agent("Am I eligible to vote if I am 20 years old?"))
print(ask_agent("Am I eligible to vote if I am 14 years old?"))


*************************************

Invoking calculator with query: {'expression': '2+2'}
Evaluating expression: 2+2
The result of 2+2 is: 4

*************************************

Invoking calculator with query: {'expression': '10 * 5'}
Evaluating expression: 10 * 5
The result of 10 * 5 is: 50

*************************************

Invoking calculator with query: {'expr': '3 + 4'}
Evaluating expression: 0
The result of 0 is: 0

*************************************

Invoking voter_helper with query: 20
Age 20 is eligible for voting.

*************************************

Invoking voter_helper with query: 14
Age 14 is not eligible for voting.
